# Which methods make the same mistakes?


**What is measured (instruction 64.32).** On three labels, all 14 recorded methods called the same genes with those genes' answers hidden (the strictly nested outer tests of 64.34). For every pair: the correlation of their errors on the genes both answered, with a 95 % interval from resampling whole orthogroups. Two methods with highly correlated errors are one piece of evidence, not two; a pair with under 100 genes in common is unresolved.

Built 2026-11-07.

In [1]:
import itertools, os, numpy as np, pandas as pd
FOLDERS = {'dtm_class': 'meta_inference_nested_2026_11_07', 'compartment': 'meta_inference_nested_2026_11_07_compartment',
           'cellcycle_phase': 'meta_inference_nested_2026_11_07_cellcycle_phase'}
nodes = pd.read_parquet(os.path.join(REPO, 'parascape/data/nodes.parquet')).set_index('gene_id')
rng = np.random.default_rng(0)
rows = []
for label, folder in FOLDERS.items():
    path = os.path.join(REPO, 'results', folder)
    calls = pd.concat([pd.read_parquet(os.path.join(path, f)).assign(strategy=f[6:-8]) for f in os.listdir(path)
                       if f.startswith('calls_') and f.endswith('.parquet')])
    calls = calls[calls.role == 'outer_test']
    wide = calls.pivot_table(index='gene_id', columns='strategy', values='call', aggfunc='first')
    truth = calls.groupby('gene_id').truth.first().reindex(wide.index)
    groups = nodes['orthogroup'].reindex(wide.index).fillna(pd.Series(wide.index, index=wide.index)).to_numpy()
    for a, b in itertools.combinations(wide.columns, 2):
        both = (wide[a] != 'ABSTAIN') & (wide[b] != 'ABSTAIN')
        n = int(both.sum())
        if n < 100:
            rows.append(dict(label=label, a=a, b=b, genes=n, error_correlation=np.nan, low=np.nan, high=np.nan, status='unresolved'))
            continue
        ea = (wide.loc[both, a] != truth[both]).to_numpy(float); eb = (wide.loc[both, b] != truth[both]).to_numpy(float)
        if ea.std() == 0 or eb.std() == 0:
            rows.append(dict(label=label, a=a, b=b, genes=n, error_correlation=np.nan, low=np.nan, high=np.nan, status='no errors to compare'))
            continue
        phi = float(np.corrcoef(ea, eb)[0, 1])
        g = groups[both.to_numpy()]; ug = np.unique(g); idx_of = {x: np.flatnonzero(g == x) for x in ug}
        boots = []
        for _ in range(200):
            idx = np.concatenate([idx_of[x] for x in rng.choice(ug, size=len(ug))])
            if ea[idx].std() and eb[idx].std():
                boots.append(np.corrcoef(ea[idx], eb[idx])[0, 1])
        rows.append(dict(label=label, a=a, b=b, genes=n, error_correlation=round(phi, 3),
                         low=round(float(np.percentile(boots, 2.5)), 3), high=round(float(np.percentile(boots, 97.5)), 3), status='measured'))
pairs = pd.DataFrame(rows)
pairs.to_csv(os.path.join(OUT, 'pairs.tsv'), sep='\t', index=False)
summary = pairs.groupby('label').agg(pairs=('a', 'size'), unresolved=('status', lambda s: int((s != 'measured').sum())),
    median_error_correlation=('error_correlation', 'median'), highest=('error_correlation', 'max')).round(3)
summary.to_csv(os.path.join(OUT, 'summary.tsv'), sep='\t')
summary

,pairs,unresolved,median_error_correlation,highest
label,,,,
cellcycle_phase,91,28,0.35,1
compartment,91,0,0.338,1
dtm_class,91,2,0.298,1


In [2]:
pairs[pairs.status == 'measured'].sort_values('error_correlation', ascending=False).head(12)

,label,a,b,genes,error_correlation,low,high,status
20,dtm_class,feature_knn,stratum_focus,8136,1,1,1,measured
90,dtm_class,triangulation,understudied_first,7772,1,1,1,measured
111,compartment,feature_knn,stratum_focus,2802,1,1,1,measured
272,cellcycle_phase,triangulation,understudied_first,729,1,1,1,measured
181,compartment,triangulation,understudied_first,2549,1,1,1,measured
202,cellcycle_phase,feature_knn,stratum_focus,860,1,1,1,measured
106,compartment,feature_knn,layer_vote,2802,0.981,0.973,0.988,measured
141,compartment,layer_vote,stratum_focus,2802,0.981,0.974,0.988,measured
197,cellcycle_phase,feature_knn,layer_vote,860,0.914,0.884,0.943,measured
232,cellcycle_phase,layer_vote,stratum_focus,860,0.914,0.887,0.939,measured
